In [237]:
import numpy as np  # numerical operations (arrays, math functions)
import pandas as pd  # loading and manipulating the dataset as a DataFrame
import matplotlib.pyplot as plt  # plotting (imported for potential visualizations)
from google.colab import files  # Colab's file-upload widget
uploaded = files.upload()  # opens a file picker so we can upload Cleaned_Hotel.csv from our computer
df = pd.read_csv("Cleaned_Hotel.csv")  # read the uploaded CSV into a DataFrame called df


Saving Cleaned_Hotel.csv to Cleaned_Hotel (2).csv


In [238]:
df.head(10)  # preview the first 10 rows to sanity-check the data loaded correctly


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,0,C,C,3,0,0,Transient,0.0,0,0
1,Resort Hotel,0,386,2015,July,27,1,0,0,2,...,0,C,C,4,0,0,Transient,0.0,0,0
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,0,A,C,0,0,0,Transient,75.0,0,0
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,0,A,A,0,0,0,Transient,75.0,0,0
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,0,A,A,0,0,0,Transient,98.0,0,1
5,Resort Hotel,0,0,2015,July,27,1,0,2,2,...,0,C,C,0,0,0,Transient,107.0,0,0
6,Resort Hotel,0,9,2015,July,27,1,0,2,2,...,0,C,C,0,0,0,Transient,103.0,0,1
7,Resort Hotel,1,85,2015,July,27,1,0,3,2,...,0,A,A,0,0,0,Transient,82.0,0,1
8,Resort Hotel,1,75,2015,July,27,1,0,3,2,...,0,D,D,0,0,0,Transient,105.5,0,0
9,Resort Hotel,1,23,2015,July,27,1,0,4,2,...,0,E,E,0,0,0,Transient,123.0,0,0


In [239]:
from sklearn.model_selection import train_test_split  # splits data into training and testing sets
from sklearn.linear_model import LinearRegression  # the linear regression model class
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # regression evaluation metrics


is_canceled is the target(y)
while deposit_type will be the independent variable on x-axis

In [288]:
print(df.columns.tolist()) #seeing the available features before using it


['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'room_mismatch', 'deposit_flag', 'is_online_ta', 'is_groups', 'is_transient', 'is_transient_party', 'is_contract']


In [312]:
df['room_mismatch'] = (df['reserved_room_type'] != df['assigned_room_type']).astype(int)  # 1 if guest got a different room than reserved, else 0
df['deposit_flag'] = (df['deposit_type'] == 2).astype(int)  # 1 if deposit_type is category 2 (the rare category with a ~95% cancellation rate), else 0
df['is_online_ta'] = (df['market_segment'] == 'Online TA').astype(int)  # 1 if booked through an Online Travel Agency, else 0
df['is_groups'] = (df['market_segment'] == 'Groups').astype(int)  # 1 if booked as part of a group, else 0
df['is_transient_party'] = (df['customer_type'] == 'Transient-Party').astype(int)  # 1 if customer_type is Transient-Party, else 0
df['is_contract'] = (df['customer_type'] == 'Contract').astype(int)  # 1 if customer_type is Contract, else 0
X = df[['is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled',
        'deposit_flag', 'lead_time', 'room_mismatch', 'is_online_ta', 'is_groups',
        'is_transient_party', 'is_contract', 'adr', 'stays_in_weekend_nights',
        'stays_in_week_nights', 'booking_changes', 'total_of_special_requests',
        'required_car_parking_spaces', 'days_in_waiting_list', 'adults', 'children', 'babies']] #can leave the inside square brackets or can remove it
# X = feature matrix: every column the model is allowed to use to make a prediction (double brackets keep it as a 2D DataFrame, which sklearn requires)
y = df['is_canceled']
# y = target: the single column we're trying to predict (1 = canceled, 0 = not canceled)

print('Rows used for the model:', len(df))  # confirms how many total rows are available before splitting


Rows used for the model: 87396


In [313]:
# 80% of the data will be used in training the model while the other 20% will be used to evaluate the model accuracy on unseen data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=19)
#random_state controls the randomness used when train_test_split shuffles and splits your data. Without it, every time you rerun the cell you'd get a different random 80/20 split.
print('Training rows:', len(X_train))  # rows the model will actually learn from
print('Testing rows:', len(X_test))  # rows held back to fairly evaluate the model afterward


Training rows: 69916
Testing rows: 17480


Linear Regression tries to find the best fit line between the feature and the target,by the function y=mx+b

Where:

m = coefficient (slope),
b = intercept,
x = 'deposit_type','previous_cancellations','previous_bookings_not_canceled',
y = predict is_canceled

In [314]:

model = LinearRegression()  # create an (untrained) linear regression model object
model.fit(X_train, y_train)  # train it: find the coefficients that best fit X_train to y_train


LinearRegression()

In [315]:
print('Coefficient:', model.coef_[0]) # NOTE: this is a MULTIPLE linear regression with 20 features, so model.coef_ is
# actually an array of 20 coefficients (one weight per feature), not a single overall coefficient.
# model.coef_[0] only prints the weight for the FIRST feature in X ('is_repeated_guest'), not "the" coefficient of the model.
print('Intercept:', model.intercept_) #the intercection point on y-axis (the predicted value when every feature is 0)
print(f'Equation: probability of cancellation = {model.coef_[0]:.2f} \u00d7 X + {model.intercept_:.2f}') # again, this equation is incomplete/misleading:
# with 20 features the real equation is y = b + (coef_0 * feature_0) + (coef_1 * feature_1) + ... + (coef_19 * feature_19),
# not a single-feature line. To see every feature's actual weight, use: for name, c in zip(X.columns, model.coef_): print(name, c)


Coefficient: 0.0029512616010804464
Intercept: 0.13274177478031782
Equation: probability of cancellation = 0.00 × X + 0.13


In [316]:
y_pred = model.predict(X_test)  # run the trained model on the unseen test rows to get raw (continuous) predictions

results = pd.DataFrame({
    'Actual': y_test.values,  # the true is_canceled labels (0 or 1)
    'Predicted': y_pred  # the model's raw continuous output (not restricted to 0 or 1)
})

results.head(10)  # preview: notice Predicted values are decimals like 0.259, not clean 0/1 labels


,Actual,Predicted
0,0,0.259133
1,1,0.365014
2,1,0.100495
3,0,0.205760
4,0,0.419646
5,0,0.291358
6,0,0.042079
7,0,0.117559
8,0,0.017296
9,0,0.037817


In [317]:
print(results['Predicted'].nunique()) #to see the frequency of non unqiue predicted values
print(results['Predicted'].value_counts())  # shows predictions are almost all unique decimal values — confirms linear regression
# outputs a continuous number, not a class, which is why we'll need to threshold it below to get 0/1 predictions


17130
Predicted
 1.054750    7
 0.169578    6
 0.179746    6
 0.195830    6
 0.146248    6
            ..
 0.441449    1
 0.411081    1
 0.330688    1
 0.206945    1
-0.114926    1
Name: count, Length: 17130, dtype: int64


In [318]:
#evaluation of the model by MSE of linear regression
mse = mean_squared_error(y_test, y_pred)  # average squared difference between raw predictions and true 0/1 labels
print('Mean Squared Error:', mse)  # this is the metric linear regression is actually designed to be judged on


Mean Squared Error: 0.15892066619848222


In [319]:
from sklearn.metrics import accuracy_score  # classification metric: % of predictions that exactly match the true label

# Predictions on both sets (continuous values, need thresholding)
y_train_pred = model.predict(X_train)  # raw continuous predictions on the training set
y_test_pred = model.predict(X_test)  # raw continuous predictions on the test set

# Threshold at 0.5 to convert into class labels
y_train_pred_class = (y_train_pred >= 0.5).astype(int)  # >= 0.5 becomes predicted class 1 (canceled), else 0
y_test_pred_class = (y_test_pred >= 0.5).astype(int)  # same thresholding applied to the test predictions

# Accuracy on both sets
train_accuracy = accuracy_score(y_train, y_train_pred_class)  # how well the model fits data it was trained on
test_accuracy = accuracy_score(y_test, y_test_pred_class)  # how well the model generalizes to unseen data (the number that matters)

print(f"Train Accuracy: {train_accuracy * 100:.2f}%")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")


Train Accuracy: 78.02%
Test Accuracy: 77.57%


In [327]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix  # classification quality metrics

y_pred_class = (y_test_pred >= 0.5).astype(int)  # thresholded test predictions (0/1), reused from the accuracy step above

precision = precision_score(y_test, y_pred_class)  # of all bookings predicted as canceled, % that were actually canceled
recall = recall_score(y_test, y_pred_class)  # of all bookings that were actually canceled, % the model correctly caught
f1 = f1_score(y_test, y_pred_class)  # harmonic mean of precision and recall — balances both into one score

print(f"Precision: {precision * 100:.2f}%")
print(f"Recall: {recall * 100:.2f}%")
print(f"F1 Score: {f1 * 100:.2f}%")
print(confusion_matrix(y_test, y_pred_class))  # [[true negatives, false positives], [false negatives, true positives]]


Precision: 74.14%
Recall: 29.29%
F1 Score: 41.99%
[[12140   495]
 [ 3426  1419]]


Logistic Regression model

In [320]:
# import SKlearn
from sklearn.linear_model import LogisticRegression  # the logistic regression model class
from sklearn.metrics import classification_report, confusion_matrix  # summary report of precision/recall/f1, and the confusion matrix


In [321]:
df['room_mismatch'] = (df['reserved_room_type'] != df['assigned_room_type']).astype(int)  # 1 if guest got a different room than reserved, else 0
df['deposit_flag'] = (df['deposit_type'] == 2).astype(int)  # 1 if deposit_type is category 2 (the ~95% cancellation rate category), else 0
df['is_online_ta'] = (df['market_segment'] == 'Online TA').astype(int)  # 1 if booked through an Online Travel Agency, else 0
df['is_groups'] = (df['market_segment'] == 'Groups').astype(int)  # 1 if booked as part of a group, else 0
df['is_transient_party'] = (df['customer_type'] == 'Transient-Party').astype(int)  # 1 if customer_type is Transient-Party, else 0
df['is_contract'] = (df['customer_type'] == 'Contract').astype(int)  # 1 if customer_type is Contract, else 0

X = df[['lead_time', 'is_repeated_guest', 'previous_cancellations',
        'previous_bookings_not_canceled', 'deposit_flag', 'adr', 'room_mismatch',
        'stays_in_weekend_nights', 'stays_in_week_nights', 'booking_changes',
        'total_of_special_requests', 'required_car_parking_spaces',
        'days_in_waiting_list', 'adults', 'children', 'babies',
        'is_online_ta', 'is_groups', 'is_transient_party', 'is_contract']]  # feature matrix (same style of features used for linear regression)
y = df['is_canceled']  # target: 1 = canceled, 0 = not canceled
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=19)  # same 80/20 split logic as before

# --- Scale features (logistic regression needs this for stable and fast convergence) ---
from sklearn.preprocessing import StandardScaler  # rescales each feature to have mean 0 and standard deviation 1
scaler = StandardScaler()  # create the scaler object
X_train_scaled = scaler.fit_transform(X_train)   # fit only on train — never let test data leak into scaling
X_test_scaled = scaler.transform(X_test)          # test set only gets transformed, not fit


In [322]:
# Create model & train it
classifier = LogisticRegression(max_iter=2000, random_state=19)  # max_iter=2000 gives the solver enough iterations to fully converge
classifier.fit(X_train_scaled, y_train)  # train on the scaled training features
y_pred = classifier.predict(X_test_scaled)  # predict classes (0/1) directly — logistic regression thresholds internally at 0.5 probability


In [323]:
from sklearn.metrics import accuracy_score  # classification metric: % of predictions that exactly match the true label

train_accuracy = accuracy_score(y_train, classifier.predict(X_train_scaled))  # accuracy on data the model was trained on
test_accuracy = accuracy_score(y_test, classifier.predict(X_test_scaled))  # accuracy on unseen data (the number that matters)

print(f"Train Accuracy: {train_accuracy * 100:.2f}%")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")


Train Accuracy: 79.53%
Test Accuracy: 79.07%


In [325]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix  # classification quality metrics

precision = precision_score(y_test, y_pred)  # of all bookings predicted as canceled, % that were actually canceled
recall = recall_score(y_test, y_pred)  # of all bookings that were actually canceled, % the model correctly caught
f1 = f1_score(y_test, y_pred)  # harmonic mean of precision and recall — balances both into one score

print(f"Precision: {precision * 100:.2f}%")
print(f"Recall: {recall * 100:.2f}%")
print(f"F1 Score: {f1 * 100:.2f}%")
print(confusion_matrix(y_test, y_pred))  # [[true negatives, false positives], [false negatives, true positives]]


Precision: 69.91%
Recall: 43.01%
F1 Score: 53.26%
[[11738   897]
 [ 2761  2084]]


In [324]:
df['is_canceled'].value_counts() #NOT RELATED TO THE MODEL -- just shows the class imbalance: 63,371 not-canceled vs 24,025 canceled


,count
is_canceled,
0,63371
1,24025
